<a href="https://colab.research.google.com/github/seanmcconney10-rgb/pythonlearnin/blob/main/week_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# The functions create_table, add_game, get_all_games, update_game are assumed to be defined in a previous cell.
# Make sure to run the cell defining these database functions (cell 1lRhEGM6Ti-c) first.

def get_non_empty(prompt):
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("Input cannot be empty. Please try again.")

def get_float(prompt, min_val=None):
    while True:
        try:
            value = float(input(prompt))
            if min_val is not None and value < min_val:
                print(f"Value must be at least {min_val}. Please try again.")
            else:
                return value
        except ValueError:
            print("Invalid input. Please enter a number.")

def get_int(prompt, min_val=None, max_val=None):
    while True:
        try:
            value = int(input(prompt))
            if min_val is not None and value < min_val:
                print(f"Value must be at least {min_val}. Please try again.")
            elif max_val is not None and value > max_val:
                print(f"Value cannot exceed {max_val}. Please try again.")
            else:
                return value
        except ValueError:
            print("Invalid input. Please enter an integer.")

def game_exists(game_id, games):
    return any(g[0] == game_id for g in games)

def display_games(games):
    print("\nYour Game Library:")
    print("-" * 50)
    if not games:
        print("No games in your library yet.")
    for g in games:
        print(f"ID: {g[0]} | {g[1]} ({g[2]}, {g[3]})")
        print(f"   Hours: {g[4]} | Completion: {g[5]}%")
    print("-" * 50)


def show_stats(games):
    if not games:
        print("No games found to generate stats.")
        return

    total_hours = sum(g[4] for g in games)
    avg_completion = sum(g[5] for g in games) / len(games)

    # Backlog (under 50%)
    backlog = [g for g in games if g[5] < 50]

    # Smart suggestion: lowest completion, then lowest hours
    suggestion = sorted(games, key=lambda g: (g[5], g[4]))[0]

    print("\n📊 Stats:")
    print(f"Total Hours Played: {total_hours}")
    print(f"Average Completion: {avg_completion:.2f}%")

    print("\n🎯 Suggested Game:")
    print(f"{suggestion[1]} ({suggestion[5]}% complete, {suggestion[4]} hrs)")

    print("\n📚 Backlog (Under 50%):")
    if backlog:
        for g in backlog:
            print(f"- {g[1]} ({g[5]}%)")
    else:
        print("No backlog games 🎉")

def main():
    create_table()

    while True:
        print("\n--- Game Library Menu ---")
        print("1. Add Game")
        print("2. View All Games")
        print("3. Update Progress")
        print("4. Show Stats")
        print("5. Delete Game")
        print("6. Filter by Genre")
        print("7. Filter by Platform")
        print("8. Exit")
        print("-------------------------")

        choice = input("Choose an option: ")

        if choice == "1":
            title = get_non_empty("Title: ")
            genre = get_non_empty("Genre: ")
            platform = get_non_empty("Platform: ")
            hours = get_float("Hours played: ", 0)
            completion = get_int("Completion %: ", 0, 100)
            add_game(title, genre, platform, hours, completion)
            print("Game added successfully!")

        elif choice == "2":
            games = get_all_games()
            display_games(games)

        elif choice == "3":
            games = get_all_games()
            if not games:
                print("No games to update.")
                continue
            display_games(games)
            game_id = get_int("Game ID to update: ", 1)
            if not game_exists(game_id, games):
                print("Game ID not found.")
            else:
                hours = get_float("New hours: ", 0)
                completion = get_int("New completion %: ", 0, 100)
                update_game(game_id, hours, completion)
                print("Game updated successfully!")

        elif choice == "4":
            games = get_all_games()
            show_stats(games)

        elif choice == "5":
            games = get_all_games()
            if not games:
                print("No games to delete.")
                continue
            display_games(games)
            game_id = get_int("Game ID to delete: ", 1)
            if not game_exists(game_id, games):
                print("Game ID not found.")
            else:
                delete_game(game_id)
                print("Game deleted successfully!")

        elif choice == "6":
            genre = get_non_empty("Enter genre to filter by: ")
            games = get_games_by_genre(genre)
            display_games(games)

        elif choice == "7":
            platform = get_non_empty("Enter platform to filter by: ")
            games = get_games_by_platform(platform)
            display_games(games)

        elif choice == "8":
            print("Exiting Game Library. Goodbye!")
            break

        else:
            print("Invalid choice. Please select a valid option (1-8).")


if __name__ == "__main__":
    main()


--- Game Library Menu ---
1. Add Game
2. View All Games
3. Update Progress
4. Show Stats
5. Delete Game
6. Filter by Genre
7. Filter by Platform
8. Exit
-------------------------
Choose an option: 1
Title: The Ballad of the signs
Genre: MMORPG, HORROR
Platform: pc
Hours played: 69
Completion %: 95
Game added successfully!

--- Game Library Menu ---
1. Add Game
2. View All Games
3. Update Progress
4. Show Stats
5. Delete Game
6. Filter by Genre
7. Filter by Platform
8. Exit
-------------------------
Choose an option: 2

Your Game Library:
--------------------------------------------------
ID: 1 | Show Stats (1. Add Game, 2)
   Hours: 3.0 | Completion: 85%
ID: 2 | The Ballad of the signs (MMORPG, HORROR, pc)
   Hours: 69.0 | Completion: 95%
--------------------------------------------------

--- Game Library Menu ---
1. Add Game
2. View All Games
3. Update Progress
4. Show Stats
5. Delete Game
6. Filter by Genre
7. Filter by Platform
8. Exit
-------------------------


In [10]:
import sqlite3

DB_NAME = "games.db"

def connect():
    return sqlite3.connect(DB_NAME)

def create_table():
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS games (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        genre TEXT,
        platform TEXT,
        hours_played REAL,
        completion INTEGER
    )
    """)

    conn.commit()
    conn.close()


def add_game(title, genre, platform, hours, completion):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO games (title, genre, platform, hours_played, completion)
    VALUES (?, ?, ?, ?, ?)
    """, (title, genre, platform, hours, completion))

    conn.commit()
    conn.close()


def get_all_games():
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM games")
    rows = cursor.fetchall()

    conn.close()
    return rows


def update_game(game_id, hours, completion):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
    UPDATE games
    SET hours_played = ?, completion = ?
    WHERE id = ?
    """, (hours, completion, game_id))

    conn.commit()
    conn.close()

def delete_game(game_id):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("DELETE FROM games WHERE id = ?", (game_id,))

    conn.commit()
    conn.close()


def get_games_by_genre(genre):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM games WHERE genre = ?", (genre,))
    rows = cursor.fetchall()

    conn.close()
    return rows


def get_games_by_platform(platform):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM games WHERE platform = ?", (platform,))
    rows = cursor.fetchall()

    conn.close()
    return rows